# Benign and Pathogenic Variant Analysis

A simple analysis of unique variants, mutation locations, targeted genes, and coding versus non-coding mutations.

In [ ]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
COLORS = {"Benign": "#4C78A8", "Pathogenic": "#E45756"}

## 1. Read the two files

In [ ]:
data_dir = Path(".")
if not (data_dir / "ClinVar_benign_data.txt").exists():
    data_dir = Path("data")

benign = pd.read_csv(data_dir / "ClinVar_benign_data.txt", sep="\t", low_memory=False)
pathogenic = pd.read_csv(data_dir / "ClinVar_pathogenic_data.txt", sep="\t", low_memory=False)

for name, df in {"Benign": benign, "Pathogenic": pathogenic}.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(df.columns.tolist(), "\n")

## 2. Remove exact duplicates and create unique tables

The files contain many transcript annotations per mutation. These are not removed as exact duplicates. Separate tables are made for one row per variation and one row per location.

In [ ]:
benign_clean = benign.drop_duplicates().copy()
pathogenic_clean = pathogenic.drop_duplicates().copy()

benign_unique_variations = benign_clean.drop_duplicates(subset="#Uploaded_variation").copy()
pathogenic_unique_variations = pathogenic_clean.drop_duplicates(subset="#Uploaded_variation").copy()

benign_unique_locations = benign_clean.drop_duplicates(subset="Location").copy()
pathogenic_unique_locations = pathogenic_clean.drop_duplicates(subset="Location").copy()

summary = pd.DataFrame({
    "Dataset": ["Benign", "Pathogenic"],
    "Original rows": [len(benign), len(pathogenic)],
    "After exact duplicates removed": [len(benign_clean), len(pathogenic_clean)],
    "Unique variations": [len(benign_unique_variations), len(pathogenic_unique_variations)],
    "Unique locations": [len(benign_unique_locations), len(pathogenic_unique_locations)]
})
summary

Use `benign_unique_variations`, `pathogenic_unique_variations`, `benign_unique_locations`, or `pathogenic_unique_locations` for later work, depending on the required unit of analysis.

## 3. Do benign and pathogenic mutations occur at the same locations?

In [ ]:
benign_locations = set(benign_unique_locations["Location"])
pathogenic_locations = set(pathogenic_unique_locations["Location"])
shared_locations = benign_locations & pathogenic_locations

print("Benign unique locations:", len(benign_locations))
print("Pathogenic unique locations:", len(pathogenic_locations))
print("Shared locations:", len(shared_locations))
print("Shared location values:", sorted(shared_locations))

In [ ]:
def location_start(location):
    return int(re.search(r":(\d+)", location).group(1))

benign_positions = benign_unique_locations["Location"].map(location_start)
pathogenic_positions = pathogenic_unique_locations["Location"].map(location_start)

fig, ax = plt.subplots(figsize=(10, 3))
ax.scatter(benign_positions, [0] * len(benign_positions), s=20, alpha=0.55, color=COLORS["Benign"], label="Benign")
ax.scatter(pathogenic_positions, [1] * len(pathogenic_positions), s=28, alpha=0.75, color=COLORS["Pathogenic"], label="Pathogenic")
ax.set_yticks([0, 1], ["Benign", "Pathogenic"])
ax.set_xlabel("Genomic position on chromosome 17")
ax.set_title("Unique mutation locations")
ax.ticklabel_format(style="plain", axis="x")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

## 4. Which genes are targeted?

Counts below use unique variation-gene pairs, so repeated transcript annotations do not inflate the result.

In [ ]:
def extract_symbol(extra):
    match = re.search(r"(?:^|;)SYMBOL=([^;]+)", str(extra))
    return match.group(1) if match else "Unknown"

benign_gene_pairs = (
    benign_clean[["#Uploaded_variation", "SYMBOL"]]
    .rename(columns={"SYMBOL": "Gene"})
    .drop_duplicates()
)

pathogenic_gene_pairs = pathogenic_clean[["#Uploaded_variation", "Extra"]].copy()
pathogenic_gene_pairs["Gene"] = pathogenic_gene_pairs["Extra"].map(extract_symbol)
pathogenic_gene_pairs = pathogenic_gene_pairs[["#Uploaded_variation", "Gene"]].drop_duplicates()

gene_counts = pd.concat([
    benign_gene_pairs["Gene"].value_counts().rename("Benign"),
    pathogenic_gene_pairs["Gene"].value_counts().rename("Pathogenic")
], axis=1).fillna(0).astype(int)
gene_counts = gene_counts.sort_values(["Pathogenic", "Benign"], ascending=False)
gene_counts

In [ ]:
genes_to_plot = gene_counts.sum(axis=1).nlargest(10).index
ax = gene_counts.loc[genes_to_plot].sort_values("Benign").plot(
    kind="barh", figsize=(8, 4), color=[COLORS["Benign"], COLORS["Pathogenic"]]
)
ax.set_xlabel("Number of unique variations")
ax.set_ylabel("Gene")
ax.set_title("Genes targeted by the mutations")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## 5. Coding versus non-coding mutations

A unique variation is called **coding** if at least one of its annotations contains a coding consequence. Otherwise, it is called **non-coding**.

In [ ]:
coding_terms = {
    "coding_sequence_variant", "missense_variant", "synonymous_variant",
    "stop_gained", "stop_lost", "start_lost", "frameshift_variant",
    "inframe_insertion", "inframe_deletion", "protein_altering_variant"
}

def has_coding_consequence(consequences):
    terms = set()
    for value in consequences.dropna():
        terms.update(str(value).split(","))
    return bool(terms & coding_terms)

def coding_summary(df):
    is_coding = df.groupby("#Uploaded_variation")["Consequence"].apply(has_coding_consequence)
    labels = is_coding.map({True: "Coding", False: "Non-coding"})
    return labels.value_counts().reindex(["Coding", "Non-coding"], fill_value=0)

coding_counts = pd.DataFrame({
    "Benign": coding_summary(benign_clean),
    "Pathogenic": coding_summary(pathogenic_clean)
})
coding_percent = coding_counts.div(coding_counts.sum(axis=0), axis=1).mul(100)
coding_counts

In [ ]:
ax = coding_percent.T.plot(
    kind="bar", stacked=True, figsize=(6, 4), color=["#59A14F", "#BAB0AC"]
)
ax.set_ylabel("Unique variations (%)")
ax.set_xlabel("")
ax.set_title("Coding and non-coding mutations")
ax.legend(title="Sequence type", frameon=False, bbox_to_anchor=(1.02, 1))
ax.set_ylim(0, 100)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

coding_percent.round(1).astype(str) + "%"